<a href="https://colab.research.google.com/github/asokone/Advanced-Python-and-Machine-Learning/blob/main/Twitter-roBERTa-base%20for%20Sentiment%20Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Twitter-roBERTa-base for Sentiment Analysis
This is a roBERTa-base model trained on ~58M tweets and finetuned for sentiment analysis with the TweetEval benchmark. This model is suitable for English (for a similar multilingual model, see XLM-T).

Reference Paper: TweetEval (Findings of EMNLP 2020).
Git Repo: Tweeteval official repository.

**Labels:** 0 -> Negative; 1 -> Neutral; 2 -> Positive

New! We just released a new sentiment analysis model trained on more recent and a larger quantity of tweets. See twitter-roberta-base-sentiment-latest and TweetNLP for more details.

Example of classification

In [ ]:
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer
import numpy as np
from scipy.special import softmax
import csv
import urllib.request

# Preprocess text (username and link placeholders)
def preprocess(text):
    new_text = []


    for t in text.split(" "):
        t = '@user' if t.startswith('@') and len(t) > 1 else t
        t = 'http' if t.startswith('http') else t
        new_text.append(t)
    return " ".join(new_text)

# Tasks:
# emoji, emotion, hate, irony, offensive, sentiment
# stance/abortion, stance/atheism, stance/climate, stance/feminist, stance/hillary

task='sentiment'
MODEL = f"cardiffnlp/twitter-roberta-base-{task}"

tokenizer = AutoTokenizer.from_pretrained(MODEL)

# download label mapping
labels=[]
mapping_link = f"https://raw.githubusercontent.com/cardiffnlp/tweeteval/main/datasets/{task}/mapping.txt"
with urllib.request.urlopen(mapping_link) as f:
    html = f.read().decode('utf-8').split("\n")
    csvreader = csv.reader(html, delimiter='\t')
labels = [row[1] for row in csvreader if len(row) > 1]

# PT
model = AutoModelForSequenceClassification.from_pretrained(MODEL)
model.save_pretrained(MODEL)

text = "Good night 😊"
text = preprocess(text)
encoded_input = tokenizer(text, return_tensors='pt')
output = model(**encoded_input)
splitted_output = output[0]
scores = splitted_output[0].detach().numpy()
scores = softmax(scores)

# # TF
# model = TFAutoModelForSequenceClassification.from_pretrained(MODEL)
# model.save_pretrained(MODEL)

# text = "Good night 😊"
# encoded_input = tokenizer(text, return_tensors='tf')
# output = model(encoded_input)
# scores = output[0][0].numpy()
# scores = softmax(scores)

ranking = np.argsort(scores)
ranking = ranking[::-1]
for i in range(scores.shape[0]):
    l = labels[ranking[i]]
    s = scores[ranking[i]]
    print(f"{i+1}) {l} {np.round(float(s), 4)}")

### Sentiment Analysis Function

Let's wrap the classification logic into a reusable function.

In [ ]:
def classify_sentiment(input_text):
    # Preprocess the input text
    processed_text = preprocess(input_text)

    # Encode the input text
    encoded_input = tokenizer(processed_text, return_tensors='pt')

    # Get model output
    output = model(**encoded_input)
    splitted_output = output[0]
    scores = splitted_output[0].detach().numpy()
    scores = softmax(scores)

    # Get the top sentiment label and its score
    ranking = np.argsort(scores)
    ranking = ranking[::-1]

    result = []
    for i in range(scores.shape[0]):
        l = labels[ranking[i]]
        s = scores[ranking[i]]
        result.append({'label': l, 'score': np.round(float(s), 4)})

    return result

print("Sentiment analysis function defined.")

### Demo: Classify a custom text input

In [ ]:
custom_text = "This is an amazing day! I am so happy! 😊"
classification_results = classify_sentiment(custom_text)

print(f"Text: '{custom_text}'")
print("Sentiment Classification:")
for res in classification_results:
    print(f"  - {res['label']}: {res['score']}")

custom_text_2 = "I'm feeling a bit down today, it's raining."
classification_results_2 = classify_sentiment(custom_text_2)

print(f"\nText: '{custom_text_2}'")
print("Sentiment Classification:")
for res in classification_results_2:
    print(f"  - {res['label']}: {res['score']}")